# Chapter 6 — Regularisation  ·  *PyTorch Companion*
**Track:** ML from Scratch · California Housing dataset

> This is the PyTorch companion to [notebook.ipynb](notebook.ipynb). All theoretical content
> and sklearn experiments are preserved verbatim. Wherever the original uses TensorFlow/Keras,
> a runnable **PyTorch equivalent** is added directly below.

## Core Idea

Regularisation closes the gap between training performance and test performance.

| Tool | Mechanism |
|---|---|
| L2 (Weight Decay) | penalise large weights — shrink all toward zero |
| L1 (Lasso) | penalise weight magnitude — push many to exactly zero |
| Dropout | randomly zero neurons during training — force redundancy |
| Early Stopping | halt when validation loss stops improving |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error

housing = fetch_california_housing()
X, y = housing.data, housing.target

# Three-way split: train / val / test
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

# Scale on train only — never fit scaler on val/test
scaler = StandardScaler()
X_tr_s  = scaler.fit_transform(X_tr)
X_val_s = scaler.transform(X_val)
X_te_s  = scaler.transform(X_te)

print(f"Train: {X_tr_s.shape}  Val: {X_val_s.shape}  Test: {X_te_s.shape}")

## Baseline — No Regularisation

Establish the overfitting reference point. We expect train R² >> test R².

In [ ]:
base = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    alpha=0.0,            # L2 strength — off
    max_iter=500,
    random_state=42,
)
base.fit(X_tr_s, y_tr)

r2_tr_base = r2_score(y_tr,  base.predict(X_tr_s))
r2_te_base = r2_score(y_te,  base.predict(X_te_s))
gap_base   = r2_tr_base - r2_te_base

print(f"Baseline — train R²={r2_tr_base:.4f}  test R²={r2_te_base:.4f}  gap={gap_base:+.4f}")
print(f"Epochs run: {base.n_iter_}")

## L2 Regularisation — Math

$$\mathcal{L}_\text{L2} = \mathcal{L}_\text{MSE} + \lambda \sum_l \|\mathbf{W}_l\|_F^2$$

Gradient: $\;\nabla_W \mathcal{L}_\text{L2} = \nabla_W \mathcal{L}_\text{MSE} + 2\lambda \mathbf{W}$

Update: $\;\mathbf{W} \leftarrow (1 - 2\eta\lambda)\mathbf{W} - \eta \nabla_W \mathcal{L}_\text{MSE}$

The factor $(1-2\eta\lambda) < 1$ **decays** the weight every step — hence "weight decay".

In [ ]:
# L2 sweep: alpha in log scale
alphas = [0.0, 1e-5, 1e-4, 1e-3, 1e-2, 0.1]
l2_results = []

for alpha in alphas:
    m = MLPRegressor(
        hidden_layer_sizes=(128, 64), activation='relu',
        solver='adam', alpha=alpha, max_iter=400, random_state=42
    ).fit(X_tr_s, y_tr)
    r2_tr = r2_score(y_tr,  m.predict(X_tr_s))
    r2_te = r2_score(y_te,  m.predict(X_te_s))
    l2_results.append({'alpha': alpha, 'train': r2_tr, 'test': r2_te, 'gap': r2_tr - r2_te})
    print(f"alpha={alpha:.0e}  train={r2_tr:.4f}  test={r2_te:.4f}  gap={r2_tr-r2_te:+.4f}")

# Plot train vs test R² across alphas
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
xs = range(len(alphas))
trains = [r['train'] for r in l2_results]
tests  = [r['test']  for r in l2_results]
gaps   = [r['gap']   for r in l2_results]
labels = [str(a) for a in alphas]

axes[0].plot(xs, trains, marker='o', label='Train R²', color='steelblue')
axes[0].plot(xs, tests,  marker='s', label='Test R²',  color='coral')
axes[0].set_xticks(xs)
axes[0].set_xticklabels(labels, rotation=30)
axes[0].set_xlabel('L2 alpha')
axes[0].set_ylabel('R²')
axes[0].set_title('L2 Alpha Sweep — R²')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(xs, gaps, color=['red' if g > 0.05 else 'green' for g in gaps], alpha=0.7)
axes[1].set_xticks(xs)
axes[1].set_xticklabels(labels, rotation=30)
axes[1].axhline(0.05, color='red', linestyle='--', linewidth=1, label='gap=0.05 threshold')
axes[1].set_xlabel('L2 alpha')
axes[1].set_ylabel('Generalisation gap (train − test R²)')
axes[1].set_title('L2 Alpha Sweep — Gap')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('L2 Regularisation Sweep on California Housing', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## L1 vs L2 — Sparsity Comparison

L1 pushes weights to exactly zero (sparse); L2 shrinks them but almost never reaches exactly zero.

We demo this on a single linear layer (sklearn Lasso vs Ridge) to isolate the effect before applying it to a neural net.

In [ ]:
from sklearn.linear_model import Lasso, Ridge

alpha_val = 0.05
lasso = Lasso(alpha=alpha_val).fit(X_tr_s, y_tr)
ridge = Ridge(alpha=alpha_val).fit(X_tr_s, y_tr)

feature_names = housing.feature_names

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, model, name, color in [
    (axes[0], lasso, f'L1 (Lasso, α={alpha_val})',  'coral'),
    (axes[1], ridge, f'L2 (Ridge, α={alpha_val})',   'steelblue'),
]:
    coefs = model.coef_
    bars = ax.bar(feature_names, coefs, color=color, alpha=0.8)
    ax.axhline(0, color='grey', linewidth=0.8)
    ax.set_title(name)
    ax.set_xlabel('Feature')
    ax.set_ylabel('Coefficient')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)
    n_zero = (np.abs(coefs) < 1e-6).sum()
    ax.set_title(f'{name}\n({n_zero}/{len(coefs)} weights = 0)')

plt.suptitle('L1 vs L2: Sparsity Effect on Coefficients', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Dropout — Manual Inverted Dropout

$$\tilde{\mathbf{h}} = \frac{1}{1-p} \cdot (\mathbf{h} \odot \mathbf{m}), \quad m_i \sim \text{Bernoulli}(1-p)$$

Scaling by $\frac{1}{1-p}$ keeps the **expected activation magnitude** the same as without dropout, so test inference (where dropout is disabled) sees the same scale it was trained with.

In [ ]:
rng = np.random.default_rng(42)

def dropout(h, p, training=True, rng=None):
    """Inverted dropout.
    h        : activation tensor, shape (n_samples, n_units)
    p        : drop probability (fraction of neurons to zero)
    training : if False, returns h unchanged (test-time behaviour)
    """
    if not training or p == 0:
        return h
    if rng is None:
        rng = np.random.default_rng()
    mask = (rng.random(h.shape) > p).astype(float)   # 1 = keep, 0 = drop
    return h * mask / (1 - p)                          # scale up survivors

# Demo: show expected value is preserved with inverted dropout
h = np.ones((10000, 64))    # all-ones activations, easy to reason about expected value
p = 0.3

h_dropped = dropout(h, p=p, training=True, rng=rng)
h_test    = dropout(h, p=p, training=False)

print(f"True activation mean:          {h.mean():.4f}")
print(f"After dropout (training):      {h_dropped.mean():.4f}  ← should ≈ 1.0")
print(f"After dropout (test, no drop): {h_test.mean():.4f}  ← should = 1.0")
print(f"Fraction of neurons zeroed:    {(h_dropped == 0).mean():.3f}  ← should ≈ {p}")

In [ ]:
# Visualise: what a layer looks like before vs after dropout
h_sample = rng.normal(0, 1, (1, 64))   # one sample, 64 neurons
h_after  = dropout(h_sample, p=0.4, training=True, rng=rng)

fig, axes = plt.subplots(2, 1, figsize=(14, 3), sharex=True)
axes[0].bar(range(64), h_sample[0], color='steelblue', alpha=0.8)
axes[0].set_title('Neuron activations — before dropout')
axes[0].set_ylabel('Activation')
axes[1].bar(range(64), h_after[0],
            color=['red' if v == 0 else 'steelblue' for v in h_after[0]], alpha=0.8)
axes[1].set_title('Neuron activations — after dropout (p=0.4, red = zeroed)')
axes[1].set_ylabel('Activation')
axes[1].set_xlabel('Neuron index')
plt.tight_layout()
plt.show()

## Early Stopping

Train a network with early stopping enabled. Track training and validation loss per epoch to see the overfitting inflection point.

In [ ]:
es_model = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=25,   # patience
    max_iter=600,
    random_state=42,
)
es_model.fit(X_tr_s, y_tr)

r2_tr_es = r2_score(y_tr, es_model.predict(X_tr_s))
r2_te_es = r2_score(y_te, es_model.predict(X_te_s))

print(f"Early stopping — stopped at epoch {es_model.n_iter_} / 600")
print(f"Train R²={r2_tr_es:.4f}  Test R²={r2_te_es:.4f}  gap={r2_tr_es-r2_te_es:+.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(es_model.loss_curve_, label='Training loss', color='steelblue')
if hasattr(es_model, 'validation_scores_'):
    ax.plot(
        [-s for s in es_model.validation_scores_],
        label='Validation loss (−R²)', color='coral', linestyle='--'
    )
ax.axvline(es_model.n_iter_ - 1, color='green', linestyle=':', linewidth=2,
           label=f'Stopped at epoch {es_model.n_iter_}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Early Stopping — Training vs Validation Loss')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Full Comparison — All Regularisation Strategies

In [ ]:
configs = [
    ('Baseline',              dict(alpha=0.0,  early_stopping=False, max_iter=400)),
    ('L2 (α=1e-4)',           dict(alpha=1e-4, early_stopping=False, max_iter=400)),
    ('L2 (α=1e-3)',           dict(alpha=1e-3, early_stopping=False, max_iter=400)),
    ('Early Stopping',        dict(alpha=0.0,  early_stopping=True,
                                   validation_fraction=0.15, n_iter_no_change=25, max_iter=600)),
    ('L2 + Early Stopping',   dict(alpha=1e-4, early_stopping=True,
                                   validation_fraction=0.15, n_iter_no_change=25, max_iter=600)),
]

rows = []
for name, kwargs in configs:
    m = MLPRegressor(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        random_state=42,
        **kwargs
    ).fit(X_tr_s, y_tr)
    r2_tr = r2_score(y_tr, m.predict(X_tr_s))
    r2_te = r2_score(y_te, m.predict(X_te_s))
    rows.append((name, r2_tr, r2_te, r2_tr - r2_te, m.n_iter_))

print(f"{'Strategy':<26} {'Train R²':>9} {'Test R²':>9} {'Gap':>8} {'Epochs':>8}")
print("-" * 65)
for row in rows:
    print(f"{row[0]:<26} {row[1]:>9.4f} {row[2]:>9.4f} {row[3]:>+8.4f} {row[4]:>8}")

# Bar chart: test R² by strategy
names = [r[0] for r in rows]
test_r2s = [r[2] for r in rows]
gaps = [r[3] for r in rows]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = ['#d9534f', '#5bc0de', '#5cb85c', '#f0ad4e', '#337ab7']
axes[0].barh(names, test_r2s, color=colors, alpha=0.85)
axes[0].set_xlabel('Test R²')
axes[0].set_title('Test R² by Regularisation Strategy')
axes[0].set_xlim(0, 1)
axes[0].axvline(test_r2s[0], color='red', linestyle='--', linewidth=1, label='Baseline')
axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)

axes[1].barh(names, gaps, color=['red' if g > 0.04 else 'green' for g in gaps], alpha=0.8)
axes[1].set_xlabel('Generalisation gap (train − test R²)')
axes[1].set_title('Generalisation Gap by Strategy')
axes[1].axvline(0.04, color='red', linestyle='--', linewidth=1, label='gap=0.04 threshold')
axes[1].legend()
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('Regularisation Strategies — California Housing', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Trap 1 — Dropout on the Output Layer

sklearn's `MLPRegressor` doesn't expose dropout directly, so we demonstrate with the manual forward pass from Ch.5.

We show that applying dropout to the **output** introduces random noise into predictions — the mean prediction is preserved but individual predictions are wildly off.

In [ ]:
# Simulate what happens if dropout is applied at the output layer
rng_trap = np.random.default_rng(42)

def relu(z): return np.maximum(0, z)

def he_init(n_in, n_out):
    return rng_trap.normal(0, np.sqrt(2 / n_in), (n_in, n_out))

W1 = he_init(8, 64);  b1 = np.zeros(64)
W2 = he_init(64, 32); b2 = np.zeros(32)
W3 = he_init(32, 1);  b3 = np.zeros(1)

def forward_correct(X):
    """Hidden dropout only (correct)."""
    h1 = dropout(relu(X @ W1 + b1), p=0.3, training=True, rng=rng_trap)
    h2 = dropout(relu(h1 @ W2 + b2), p=0.3, training=True, rng=rng_trap)
    return (h2 @ W3 + b3).ravel()

def forward_wrong(X):
    """Output dropout too (wrong for regression)."""
    h1 = dropout(relu(X @ W1 + b1), p=0.3, training=True, rng=rng_trap)
    h2 = dropout(relu(h1 @ W2 + b2), p=0.3, training=True, rng=rng_trap)
    out = (h2 @ W3 + b3).ravel()
    return dropout(out.reshape(1, -1), p=0.5, training=True, rng=rng_trap).ravel()

n_repeats = 50
sample = X_te_s[:1]   # single house

correct_preds = [forward_correct(sample)[0] for _ in range(n_repeats)]
wrong_preds   = [forward_wrong(sample)[0]   for _ in range(n_repeats)]

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, preds, label, color in [
    (axes[0], correct_preds, 'Hidden dropout only (correct)',    'steelblue'),
    (axes[1], wrong_preds,   'Output dropout too (wrong)',       'red'),
]:
    ax.hist(preds, bins=15, color=color, alpha=0.75)
    ax.axvline(np.mean(preds), color='black', linestyle='--', label=f'mean={np.mean(preds):.2f}')
    ax.set_title(label)
    ax.set_xlabel('Predicted house value ($100k)')
    ax.set_ylabel('Count')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Trap: Dropout on Output Layer — 50 forward passes, same input',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Correct std dev: {np.std(correct_preds):.4f}")
print(f"Wrong   std dev: {np.std(wrong_preds):.4f}   ← much more variance")

## Trap 2 — Validation Contamination (Scaler Leak)

Fitting `StandardScaler` on train + val leaks test-set statistics into the model, giving an artificially optimistic validation score.

In [ ]:
# Wrong: fit scaler on all data (including val/test)
scaler_leak = StandardScaler()
X_all_s_leak = scaler_leak.fit_transform(X)   # using all 20640 rows!

# Correct: fit scaler only on training data
scaler_ok   = StandardScaler().fit(X_tr)

# Compare what the validation set looks like under each scaler
_, X_val_indices = train_test_split(
    np.arange(len(X_tmp)), test_size=0.5, random_state=42
)  # val indices in the original X

# Compare mean and std of validation features
X_val_leaked  = scaler_leak.transform(X_val)
X_val_correct = scaler_ok.transform(X_val)

print("Feature means of scaled validation set:")
print(f"  Leaked scaler:   {X_val_leaked.mean(axis=0).round(4)}")
print(f"  Correct scaler:  {X_val_correct.mean(axis=0).round(4)}")
print("\nIf val means are closer to 0 with the leaked scaler,")
print("it has seen the val distribution → artificially low loss on val.")

## Exercises

**Exercise 1 — L2 + Dropout combination**  
sklearn's `MLPRegressor` doesn't support dropout natively. Use `tensorflow.keras` to build a network with `Dropout(0.3)` after each hidden layer and `kernel_regularizer=l2(1e-4)`. Compare test R² to the sklearn L2 model.

**Exercise 2 — Patience sensitivity**  
Run early stopping with `n_iter_no_change` in `[5, 10, 20, 50]`. Plot the epoch at which each stops and its test R². Is there a point of diminishing returns?

**Exercise 3 — Weight distribution under L2**  
After fitting models with `alpha` in `[0, 1e-4, 1e-2]`, access `model.coefs_[0]` (first-layer weights). Plot histograms of the weight distributions side by side. How does increasing α change the shape and spread?

In [ ]:
# Exercise 1 scaffold — Dropout via Keras
# Requires: pip install tensorflow
# from tensorflow import keras
# from tensorflow.keras import layers
# from tensorflow.keras.regularizers import l2
#
# model = keras.Sequential([
#     layers.Dense(128, activation='relu', kernel_regularizer=l2(1e-4), input_shape=(8,)),
#     layers.Dropout(0.3),
#     layers.Dense(64, activation='relu', kernel_regularizer=l2(1e-4)),
#     layers.Dropout(0.3),
#     layers.Dense(1),   # linear output for regression
# ])
# model.compile(optimizer='adam', loss='mse')
# model.fit(X_tr_s, y_tr, epochs=200, validation_data=(X_val_s, y_val), verbose=0)
# TODO: compute and print test R²

### PyTorch equivalent — Exercise 1 (Dropout + L2)

Key API mappings vs the Keras scaffold above:

| Keras | PyTorch |
|---|---|
| `layers.Dense(n, activation='relu')` | `nn.Linear(in, n)` + `nn.ReLU()` (or `F.relu` in `forward`) |
| `layers.Dropout(0.3)` | `nn.Dropout(0.3)` — **only active in `model.train()` mode** |
| `kernel_regularizer=l2(1e-4)` | `weight_decay=2e-4` on the optimizer (note: Keras `l2(λ)` adds `λ·‖W‖²`, PyTorch `weight_decay=wd` adds `wd/2·‖W‖²`, so `wd = 2λ` for an exact match) |
| `model.compile(optimizer='adam', loss='mse')` | `optim.Adam(...)` + `nn.MSELoss()` |
| `model.fit(..., validation_data=...)` | explicit loop: zero grad → forward → loss → backward → step |
| Dropout auto-disabled at inference | **must call `model.eval()`** before `predict` |

In [ ]:
# Exercise 1 — Dropout + L2 in PyTorch (runnable)
# Requires: pip install torch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class RegressorWithDropout(nn.Module):
    """Keras analogue: Dense(128)→Dropout(0.3)→Dense(64)→Dropout(0.3)→Dense(1)."""

    def __init__(self, in_features=8, p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128), nn.ReLU(), nn.Dropout(p),
            nn.Linear(128, 64),          nn.ReLU(), nn.Dropout(p),
            nn.Linear(64, 1),            # linear output for regression
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


# Tensors on device
Xtr = torch.tensor(X_tr_s,  dtype=torch.float32, device=device)
ytr = torch.tensor(y_tr,    dtype=torch.float32, device=device)
Xvl = torch.tensor(X_val_s, dtype=torch.float32, device=device)
yvl = torch.tensor(y_val,   dtype=torch.float32, device=device)
Xte = torch.tensor(X_te_s,  dtype=torch.float32, device=device)
yte = torch.tensor(y_te,    dtype=torch.float32, device=device)

loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=256, shuffle=True)

model = RegressorWithDropout(in_features=X_tr_s.shape[1], p=0.3).to(device)
loss_fn = nn.MSELoss()
# Keras l2(1e-4) ≈ PyTorch weight_decay=2e-4 (different constant factors — see table above)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=2e-4)

EPOCHS = 200
history = {"train": [], "val": []}
for epoch in range(EPOCHS):
    model.train()                            # enables Dropout
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()

    model.eval()                             # disables Dropout
    with torch.no_grad():
        history["train"].append(loss_fn(model(Xtr), ytr).item())
        history["val"].append(loss_fn(model(Xvl), yvl).item())

# Test R² — compute without sklearn to stay pure-torch
model.eval()
with torch.no_grad():
    pred_te = model(Xte)
    ss_res = torch.sum((yte - pred_te) ** 2)
    ss_tot = torch.sum((yte - yte.mean()) ** 2)
    r2_te_torch = (1 - ss_res / ss_tot).item()

print(f"PyTorch (Dropout 0.3 + weight_decay=2e-4)  test R² = {r2_te_torch:.4f}")
print(f"Final train MSE = {history['train'][-1]:.4f}   val MSE = {history['val'][-1]:.4f}")

In [ ]:
# Exercise 2 scaffold — Patience sensitivity
patience_values = [5, 10, 20, 50]
patience_results = []

for p in patience_values:
    m = MLPRegressor(
        hidden_layer_sizes=(128, 64), activation='relu', solver='adam', alpha=1e-4,
        early_stopping=True, validation_fraction=0.15,
        n_iter_no_change=p, max_iter=600, random_state=42
    ).fit(X_tr_s, y_tr)
    r2_te = r2_score(y_te, m.predict(X_te_s))
    patience_results.append((p, m.n_iter_, r2_te))
    print(f"patience={p:3d}  stopped_at={m.n_iter_:4d}  test_R²={r2_te:.4f}")

# TODO: plot epoch stopped vs patience, and R² vs patience

In [ ]:
# Exercise 3 scaffold — Weight distribution under increasing L2
alphas_dist = [0, 1e-4, 1e-2]
fig, axes = plt.subplots(1, 3, figsize=(13, 3))

for ax, alpha in zip(axes, alphas_dist):
    m = MLPRegressor(
        hidden_layer_sizes=(128, 64), activation='relu',
        solver='adam', alpha=alpha, max_iter=300, random_state=42
    ).fit(X_tr_s, y_tr)
    weights = m.coefs_[0].ravel()   # first-layer weights flattened
    ax.hist(weights, bins=40, color='steelblue', alpha=0.75)
    ax.set_title(f'alpha={alpha}')
    ax.set_xlabel('Weight value')
    ax.set_ylabel('Count')
    ax.set_xlim(-1.5, 1.5)
    ax.grid(alpha=0.3)
    # TODO: annotate with std of weights

plt.suptitle('Weight Distributions Under Different L2 Strengths', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()